In [1]:
import numpy as np
import pandas as pd
import re
import string
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
movies = pd.read_csv('ml-latest\\movies.csv')
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [3]:
def clean_title(title):
    title = title.lower()
    #title = title.title()
    title = re.sub(f"[{string.punctuation}]", "", title)
    title = re.sub('[^a-zA-Z0-9\s]', '', title)
    return title

In [4]:
movies['Clean_Title'] = movies.title.apply(clean_title)
movies.head()

,movieId,title,genres,Clean_Title
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,toy story 1995
1,2,Jumanji (1995),Adventure|Children|Fantasy,jumanji 1995
2,3,Grumpier Old Men (1995),Comedy|Romance,grumpier old men 1995
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,waiting to exhale 1995
4,5,Father of the Bride Part II (1995),Comedy,father of the bride part ii 1995


In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer(stop_words='english', ngram_range=(1, 2))
tfidf = vectorizer.fit_transform(movies['Clean_Title'])

In [6]:
from sklearn.metrics.pairwise import cosine_similarity

def search_title(title):    
    title = clean_title(title) 
    query_vector = vectorizer.transform([title])
    similarity = cosine_similarity(query_vector, tfidf).flatten()
    indices = np.argpartition(similarity, -10)[-10:]
    result = movies.iloc[indices].sort_values('movieId', ascending=True)    
    return result

In [7]:
import ipywidgets as widgets 
from IPython.display import display

movie_input = widgets.Text(
    value="",
    placeholder='Type a movie title',
    description='Movie Title:',
    disabled=False
)

movie_list = widgets.Output()

def on_type(data):
    with movie_list:
        movie_list.clear_output()
        title = data['new']
        if len(title) >= 5:
            display(search_title(title))
           
movie_input.observe(on_type, names='value')
display(movie_input, movie_list)

Text(value='', description='Movie Title:', placeholder='Type a movie title')

Output()

In [8]:
ratings = pd.read_csv('ml-latest\\ratings.csv')
ratings.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,1225734739
1,1,110,4.0,1225865086
2,1,158,4.0,1225733503
3,1,260,4.5,1225735204
4,1,356,5.0,1225735119


In [ ]:
# Find users who rated movieId 1 with a rating of 5 or higher to identify similar users
similar_users = ratings[(ratings['movieId'] == 1) & (ratings['rating'] >= 5)]['userId'].unique()
similar_users

array([     2,     12,     51, ..., 330947, 330951, 330955],
      shape=(18331,))

In [ ]:
# the following code finds the movies that similar users have rated 4 or higher, counts how many similar users rated each movie,
# and filters for movies that at least 10% of similar users rated highly. This gives a list of recommended movies based on the preferences of users who liked movieId 1.
similar_users_rec = ratings[(ratings['userId'].isin(similar_users)) & (ratings['rating'] >= 4)]['movieId']
similar_users_rec = similar_users_rec.value_counts()/len(similar_users)
similar_users_rec = similar_users_rec[similar_users_rec >= 0.1]
similar_users_rec 

movieId
1        1.000000
260      0.491899
318      0.480388
356      0.472424
296      0.420981
           ...   
1127     0.101086
1282     0.100922
72998    0.100376
89745    0.100104
292      0.100104
Name: count, Length: 237, dtype: float64

In [ ]:
# Now, we can find all the ratings for the recommended movies from all users, not just the similar users.
# This will allow us to see how other users have rated these recommended movies.
all_users = ratings[(ratings['movieId'].isin(similar_users_rec.index)) & (ratings['rating'] >= 4)]
all_users

,userId,movieId,rating,timestamp
0,1,1,4.0,1225734739
1,1,110,4.0,1225865086
3,1,260,4.5,1225735204
4,1,356,5.0,1225735119
6,1,596,4.0,1225733524
...,...,...,...,...
33832094,330975,3996,4.0,1091585028
33832095,330975,4011,4.0,1091684285
33832111,330975,4993,4.0,1091583313
33832149,330975,6874,4.0,1091684274


In [12]:
all_users_rec = all_users['movieId'].value_counts() / len(all_users['userId'].unique())
all_users_rec


movieId
318     0.371255
296     0.299003
356     0.290958
2571    0.288649
593     0.279460
          ...   
1907    0.032142
1282    0.031114
2761    0.030041
596     0.026406
2078    0.024234
Name: count, Length: 237, dtype: float64

In [13]:
rec_percentage = pd.concat([similar_users_rec, all_users_rec], axis=1)
rec_percentage.columns = ['similar', 'all']
rec_percentage

,similar,all
movieId,,
1,1.000000,0.177437
260,0.491899,0.255377
318,0.480388,0.371255
356,0.472424,0.290958
296,0.420981,0.299003
...,...,...
1127,0.101086,0.041142
1282,0.100922,0.031114
72998,0.100376,0.066927


In [14]:
rec_percentage['score'] = rec_percentage['similar'] / rec_percentage['all']
rec_percentage.sort_values('score', ascending=False, inplace=True)
rec_percentage

,similar,all,score
movieId,,,
1,1.000000,0.177437,5.635806
3114,0.362610,0.074410,4.873108
2355,0.221701,0.048099,4.609226
2078,0.102177,0.024234,4.216269
2081,0.131744,0.032714,4.027123
...,...,...,...
4878,0.102013,0.086908,1.173804
4973,0.135999,0.119229,1.140653
109487,0.115597,0.106209,1.088389


In [15]:
rec_percentage.head(10).merge(movies, left_index=True, right_on='movieId')


,similar,all,score,movieId,title,genres,Clean_Title
0,1.000000,0.177437,5.635806,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,toy story 1995
3021,0.362610,0.074410,4.873108,3114,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy,toy story 2 1999
2264,0.221701,0.048099,4.609226,2355,"Bug's Life, A (1998)",Adventure|Animation|Children|Comedy,bugs life a 1998
1989,0.102177,0.024234,4.216269,2078,"Jungle Book, The (1967)",Animation|Children|Comedy|Musical,jungle book the 1967
1992,0.131744,0.032714,4.027123,2081,"Little Mermaid, The (1989)",Animation|Children|Comedy|Musical|Romance,little mermaid the 1989
588,0.102395,0.026406,3.877753,596,Pinocchio (1940),Animation|Children|Fantasy|Musical,pinocchio 1940
14815,0.180187,0.046475,3.877072,78499,Toy Story 3 (2010),Adventure|Animation|Children|Comedy|Fantasy|IMAX,toy story 3 2010
586,0.140963,0.038033,3.706323,594,Snow White and the Seven Dwarfs (1937),Animation|Children|Drama|Fantasy|Musical,snow white and the seven dwarfs 1937
1818,0.118160,0.032142,3.676170,1907,Mulan (1998),Adventure|Animation|Children|Comedy|Drama|Musi...,mulan 1998
1005,0.132399,0.037451,3.535270,1028,Mary Poppins (1964),Children|Comedy|Fantasy|Musical,mary poppins 1964


In [16]:
def find_similar_movies(movie_id):
    
    similar_users = ratings[(ratings['movieId'] == movie_id) & (ratings['rating'] >= 5)]['userId'].unique()
    similar_users_rec = ratings[(ratings['userId'].isin(similar_users)) & (ratings['rating'] >= 4)]['movieId']
    similar_users_rec = similar_users_rec.value_counts()/len(similar_users)
    similar_users_rec = similar_users_rec[similar_users_rec >= 0.1]

    all_users = ratings[(ratings['movieId'].isin(similar_users_rec.index)) & (ratings['rating'] >= 4)]
    all_users_rec = all_users['movieId'].value_counts() / len(all_users['userId'].unique())
    
    rec_percentage = pd.concat([similar_users_rec, all_users_rec], axis=1)
    rec_percentage.columns = ['similar', 'all']
    rec_percentage['score'] = rec_percentage['similar'] / rec_percentage['all']
    rec_percentage.sort_values('score', ascending=False, inplace=True)
    return rec_percentage.head(10).merge(movies, left_index=True, right_on='movieId')

In [17]:
movie_name_input = widgets.Text(
    value="",
    placeholder='Type a movie title',
    description='Movie Title:',
    disabled=False
)

recommendation_list = widgets.Output()

def on_type(data):
    with recommendation_list:
        recommendation_list.clear_output()
        title = data['new']
        if len(title) >= 5:
            search_results = search_title(title)
            if not search_results.empty:
                movie_id = search_results.iloc[0]['movieId']
                recommendations = find_similar_movies(movie_id)
                display(recommendations)
            else:
                print("No matching movie found.")

movie_name_input.observe(on_type, names='value')
display(movie_name_input, recommendation_list)


Text(value='', description='Movie Title:', placeholder='Type a movie title')

Output()